# A–B · Audit baseline và kiểm chứng query

**Mục tiêu:** khóa kết quả 24 caption cũ và chuẩn bị 40 truy vấn mới trước khi xem path XAI. Chạy CPU local hoặc Colab. `source_group` là nhãn từ bundle, chưa phải nhãn direct/relational đã kiểm chứng. Ảnh âm là ứng viên cùng đối tượng, chưa phải ground truth. Không ghi đè snapshot cũ.

In [ ]:
from pathlib import Path
import hashlib, sys
import pandas as pd
ROOT = Path('/content/drive/MyDrive') if Path('/content/drive/MyDrive/graphml-project3').is_dir() else next(p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'graphml-project3').is_dir())
DUONG = ROOT/'graphml-project3/duong'
sys.path.insert(0,str(DUONG))
from xai.review_workbook import baseline_review_pack, create_review_workbook
from xai.image_download import fetch_review_image
from xai.query_registry import validated_registry_rows, import_reviewed_queries
SNAP = DUONG/'runs/qa_vg_psg_xai/xai-20261121-130842'
REG = DUONG/'review/queries_relational_candidates.csv'
assert SNAP.is_dir() and REG.is_file()
print('Frozen registry SHA256:',hashlib.sha256(REG.read_bytes()).hexdigest())


## 1 · Audit snapshot cũ

Kiểm tra 216 dòng, 12 query/bộ × 3 seed × 3 mức k. Đây là leave-one-edge-out; path validity và evidence được báo riêng. Mẫu số sparsity cũ chỉ là vùng sàng lọc.

In [ ]:
old=pd.read_csv(SNAP/'analysis/xai_results_corrected.csv')
assert len(old)==216 and old.groupby('dataset').caption_id.nunique().to_dict()=={'vg':12,'psg':12}
display(old[old.requested_k==3].groupby('dataset').agg(n=('caption_id','size'),query_count=('caption_id','nunique'),screen_coverage=('screen_coverage','mean'),final_drop=('final_score_drop','mean'),random_drop=('random_final_score_drop_mean','mean'),evidence_known=('evidence_known_rate','mean')))


## 2 · Cổng query/ảnh

Bộ ứng viên có 10 `direct` nguồn và 10 `indirect` nguồn mỗi dataset. Hai người xem ảnh dương và ảnh âm khó, điền claim subject/predicate/object, `verified_group`, relevance, đối tượng chung/quan hệ khác và căn cứ ảnh. Duyệt CSV chỉ sau khi đối chiếu Excel; `validated_registry_rows` chặn mọi query chưa qua cổng. Embedding CLIP và concepts được lấy theo caption ID của chính bundle.

In [ ]:
registry=pd.read_csv(REG,keep_default_na=False)
assert len(registry)==40
display(registry.groupby(['dataset','source_group']).size())
display(registry[['dataset','query_id','query','gt_image_id','hard_negative_candidate_id','gt_source_url','hard_negative_source_url','review_status']].head(10))
try:
    print('Approved:',len(validated_registry_rows(REG)))
except ValueError as exc:
    print('Visual verification pending:',exc)


## 3 · Excel kiểm chứng

Workbook baseline nhúng 10 cặp ảnh GT/top-1 đã tải và các cạnh mô hình chọn. Workbook query mới chứa URL và nhúng ảnh khi tải được. Hai reviewer chấm độc lập; đưa bản đã hòa giải vào `REVIEWED_WORKBOOK`, nhập thành CSV mới và giữ registry nguồn bất biến.

In [ ]:
baseline_review_pack(SNAP,DUONG/'review/baseline_case_review_10.xlsx')
DOWNLOAD_QUERY_IMAGES=Path('/content').is_dir()
ALLOW_UNVERIFIED_TLS=True  # public dataset images only; each fallback is recorded in the manifest
image_dir=DUONG/'review/query_images'; image_dir.mkdir(exist_ok=True)
cases=[]; download_records=[]
for r in registry.to_dict('records'):
    case={'dataset':r['dataset'],'query_id':r['query_id'],'query':r['query'],'source_group':r['source_group'],'gt_image_id':r['gt_image_id'],'retrieved_image_id':r['hard_negative_candidate_id'],'retrieval_outcome':'pending visual review','explanation_level':'pre-XAI candidate','path_or_edge_text':'annotate claim before XAI','gt_source_url':r['gt_source_url'],'retrieved_source_url':r['hard_negative_source_url'],'review_status':'pending'}
    if DOWNLOAD_QUERY_IMAGES:
        for role,url_key,file_key in [('gt','gt_source_url','gt_image_file'),('negative','retrieved_source_url','retrieved_image_file')]:
            path=image_dir/f"{r['dataset']}_{r['query_id']}_{role}.jpg"
            try:
                info=fetch_review_image(case[url_key],path,allow_unverified_tls=ALLOW_UNVERIFIED_TLS)
                case[file_key]=str(path)
                download_records.append({'dataset':r['dataset'],'query_id':r['query_id'],'role':role,**info})
            except Exception as exc:
                download_records.append({'dataset':r['dataset'],'query_id':r['query_id'],'role':role,'source_url':case[url_key],'status':'failed','error':str(exc)})
                print('Image unavailable:',r['query_id'],role,str(exc)[:180])
    cases.append(case)
create_review_workbook(cases,[],DUONG/'review/query_candidates_40.xlsx',DUONG)
print(DUONG/'review/baseline_case_review_10.xlsx',DUONG/'review/query_candidates_40.xlsx')
if DOWNLOAD_QUERY_IMAGES:
    pd.DataFrame(download_records).to_csv(DUONG/'review/image_download_manifest.csv',index=False)
    downloaded=sum(record['status']!='failed' for record in download_records)
    print('Downloaded/available images:',downloaded,'/',len(download_records))
    if downloaded==0: raise RuntimeError('No images could be downloaded; inspect image_download_manifest.csv')
REVIEWED_WORKBOOK=None  # Path('.../adjudicated_query_review.xlsx')
if REVIEWED_WORKBOOK:
    verified=import_reviewed_queries(REG,REVIEWED_WORKBOOK,DUONG/'review/queries_relational_verified.csv')
    print('Verified query count:',len(validated_registry_rows(verified)))
